In [ ]:

import os
import pandas as pd
from ytmusicapi import YTMusic


In [ ]:
yt = YTMusic('headers_auth.json')
search_results = pd.DataFrame(yt.search("Shine on you crazy diamond"))
search_results.head(3)


In [ ]:
PLAYLIST_SONG_LIMIT=5000
PLAYLIST_LIMIT=500

In [ ]:
playlists = pd.DataFrame(yt.get_library_playlists(limit=PLAYLIST_LIMIT) )
playlists

In [ ]:
PLAYLIST_SONG_LIMIT=5000
USER='Jake G'
REMOVE_DISLIKE = True
BACKUP_DIR  = './playlists/'
PLAYLIST_TSV_COLS = ['title', 'artist', 'album', 'likeStatus', 'duration', 'videoId', 'albumId', 'artistId']

metadata = []
for i, row in playlists.iterrows():   
    print('\n\n%s (%d/%d) Playlist: %s %s' % (40*'*', i+1, len(playlists), row['title'], 40*'*'))
    # if row['playlistId'] == 'LM':
    #     continue
    if i < 175:
        continue

    # Fetch playlist
    playlist_meta = yt.get_playlist(row['playlistId'], limit=PLAYLIST_SONG_LIMIT)
    playlist_meta.pop('thumbnails', None)
    tracks = playlist_meta.pop('tracks', None)
    metadata.append(playlist_meta)
    print(pd.DataFrame.from_dict(playlist_meta, orient='index'))
    if playlist_meta['trackCount'] == 0:
        print('Skipping: %s, due to zero tracks' % playlist_meta['title'])
        continue

    # Parse playlist tracks
    tracks = pd.DataFrame(tracks)
    if REMOVE_DISLIKE:
        try:
            tracks_disliked = tracks.loc[tracks['likeStatus'] == 'DISLIKE']
            if len(tracks_disliked) and playlist_meta['author']['name'] == USER:
                print('Removing %d tracks:\n%s' % (len(tracks_disliked), tracks_disliked['title']))
                yt.remove_playlist_items(playlist_meta['id'], tracks_disliked.to_dict('records'))  
                tracks = tracks.loc[tracks['likeStatus'] != 'DISLIKE']
        except Exception as e:
            print('\nFailed to remove dislikes...\n%s\n' % e) 

    tracks['artistId'] = tracks['artists'].dropna().apply(lambda x: x[0]['id']) # TODO handle > 1 artist
    tracks['artist'] = tracks['artists'].dropna().apply(lambda x: x[0]['name'])
    tracks['albumId'] = tracks['album'].dropna().apply(lambda x: x['id'])
    tracks['album'] = tracks['album'].dropna().apply(lambda x: x['name'])
    tracks = tracks[PLAYLIST_TSV_COLS]
    tracks.to_csv(os.path.join(BACKUP_DIR, '%s.tsv' % playlist_meta['title']), sep='\t', header=True)



In [ ]:

METADATA_TSV = '_metadata.tsv'
METADATA_TSV_COLS = ['title','trackCount','duration','privacy','id']

metadata =pd.DataFrame(metadata)[METADATA_TSV_COLS]
metadata.to_csv(os.path.join(BACKUP_DIR, METADATA_TSV), sep='\t', header=True)



In [ ]:
"""
TODO
* rate thumbs up playlists ALL to LIKE
* rate thumbs down playlists ALL to dislike (except if like)


"""
